# Tahap 2: Unduh PDF Putusan

Notebook ini mengunduh file PDF putusan berdasarkan URL yang ada di metadata CSV dari tahap 1.

- **Input:** `../data/raw/metadata/metadata_*.csv` (hasil Tahap 1)
- **Output:** `../data/raw/pdf/*.pdf` + log unduhan CSV

> Jalankan **Tahap 1** terlebih dahulu untuk menghasilkan file metadata CSV.

In [ ]:
!pip install requests beautifulsoup4 lxml pandas tqdm -q
print('Instalasi selesai!')

## Konfigurasi

In [ ]:
import requests
import pandas as pd
import time
import os
import re
import glob
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

BASE_URL = 'https://putusan3.mahkamahagung.go.id'
METADATA_DIR = '../data/raw/metadata'
PDF_DIR      = '../data/raw/pdf'
os.makedirs(PDF_DIR, exist_ok=True)

# ── Parameter yang dapat disesuaikan ─────────────────────────
MAX_PDF     = 10    # Maksimal PDF yang diunduh (0 = tidak ada batas)
DELAY_DETIK = 2.0   # Delay antar request (detik)
# ─────────────────────────────────────────────────────────────

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8',
    'Referer': 'https://putusan3.mahkamahagung.go.id/'
}

print('Konfigurasi dimuat.')
print('PDF output : ' + os.path.abspath(PDF_DIR))
print('Max PDF    : ' + (str(MAX_PDF) if MAX_PDF > 0 else 'tidak ada batas'))

## Muat Metadata CSV

In [ ]:
# Cari semua CSV metadata yang tersedia
csv_files = sorted(glob.glob(os.path.join(METADATA_DIR, 'metadata_*.csv')))

if not csv_files:
    print('Tidak ada file metadata CSV di ' + METADATA_DIR)
    print('Jalankan notebook 01_scraping_metadata.ipynb terlebih dahulu.')
else:
    print('File metadata tersedia:')
    for i, f in enumerate(csv_files):
        print('  [' + str(i) + '] ' + os.path.basename(f))
    print()
    # Gunakan file terbaru secara default
    try:
        idx = int(input('Pilih nomor file [default=' + str(len(csv_files)-1) + ']: ').strip() or str(len(csv_files)-1))
    except ValueError:
        idx = len(csv_files) - 1
    INPUT_CSV = csv_files[idx]
    df_meta = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
    print('Dimuat: ' + INPUT_CSV)
    print('Total  : ' + str(len(df_meta)) + ' baris')
    display(df_meta.head())

## Fungsi Unduh PDF

In [ ]:
def ambil_halaman(url, retries=3):
    for i in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            if resp.status_code == 200:
                return BeautifulSoup(resp.content, 'lxml')
        except requests.RequestException as e:
            print('    Error (' + str(i+1) + '/' + str(retries) + '): ' + str(e))
            time.sleep(DELAY_DETIK * 2)
    return None


def ambil_url_pdf(url_detail):
    soup = ambil_halaman(url_detail)
    if not soup:
        return None
    for link in soup.find_all('a', href=True):
        href = link['href']
        teks = link.get_text(strip=True).lower()
        if href.endswith('.pdf') or 'pdf' in href.lower() or 'download' in teks:
            return BASE_URL + href if href.startswith('/') else href
    for tag in soup.find_all(attrs={'data-pdf': True}):
        return tag['data-pdf']
    return None


def unduh_pdf(url_pdf, nama_file, folder):
    path_file = os.path.join(folder, nama_file)
    if os.path.exists(path_file):
        print('    Sudah ada, skip: ' + nama_file)
        return path_file
    try:
        resp = requests.get(url_pdf, headers=HEADERS, timeout=60, stream=True)
        if resp.status_code == 200:
            with open(path_file, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return path_file
    except Exception as e:
        print('    Gagal unduh: ' + str(e))
    return None


def buat_nama_file(teks, max_len=60):
    aman = re.sub(r'[<>:"/\\|?*\s]+', '_', str(teks))
    return aman[:max_len]


print('Fungsi unduh PDF siap.')

## Jalankan Unduhan PDF

In [ ]:
print('=' * 60)
print('  MULAI UNDUH PDF')
print('=' * 60)

target_rows = df_meta.to_dict('records')
if MAX_PDF > 0:
    target_rows = target_rows[:MAX_PDF]

print('Target: ' + str(len(target_rows)) + ' PDF dari ' + str(len(df_meta)) + ' putusan')
print()

log_unduhan = []
berhasil = 0
gagal    = 0
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

for i, row in enumerate(tqdm(target_rows, desc='Unduh PDF'), 1):
    url_detail = str(row.get('url_detail', ''))
    nomor      = str(row.get('nomor', 'putusan_' + str(i)))

    if not url_detail or url_detail == 'nan':
        gagal += 1
        continue

    print('\n  [' + str(i) + '/' + str(len(target_rows)) + '] ' + nomor[:60])

    url_pdf = ambil_url_pdf(url_detail)
    time.sleep(DELAY_DETIK)

    if not url_pdf:
        print('    URL PDF tidak ditemukan')
        gagal += 1
        log_unduhan.append({**row, 'url_pdf': '', 'path_pdf': '', 'status': 'tidak_ada_link'})
        continue

    nama_pdf = buat_nama_file(nomor) + '.pdf'
    path_pdf = unduh_pdf(url_pdf, nama_pdf, PDF_DIR)
    time.sleep(DELAY_DETIK)

    if path_pdf:
        ukuran = os.path.getsize(path_pdf)
        print('    Tersimpan: ' + nama_pdf + ' (' + str(round(ukuran/1024, 1)) + ' KB)')
        berhasil += 1
        log_unduhan.append({**row, 'url_pdf': url_pdf, 'path_pdf': path_pdf, 'status': 'berhasil'})
    else:
        gagal += 1
        log_unduhan.append({**row, 'url_pdf': url_pdf, 'path_pdf': '', 'status': 'gagal_unduh'})

print()
print('=' * 60)
print('  RINGKASAN UNDUHAN')
print('  Berhasil : ' + str(berhasil) + ' file')
print('  Gagal    : ' + str(gagal) + ' file')
print('  Folder   : ' + os.path.abspath(PDF_DIR))
print('=' * 60)

if log_unduhan:
    df_log = pd.DataFrame(log_unduhan)
    path_log = os.path.join(METADATA_DIR, 'log_unduhan_' + ts + '.csv')
    df_log.to_csv(path_log, index=False, encoding='utf-8-sig')
    print('Log disimpan: ' + path_log)